# Phase I – Data Collection & Cleaning
Downloads ETF prices (yfinance) and macro indicators (ALFRED/fredapi),
cleans and aligns to month-end, saves to disk.

Outputs:
- `cleaned_prices.csv` — monthly ETF prices
- `cleaned_prices_daily.csv` — daily ETF prices (for 60-day volatility in Phase III)
- `cleaned_macro.csv` — monthly macro indicators including TB3MS for Faber cash returns

In [1]:
import urllib.request
import json
import yfinance as yf
import pandas as pd
import numpy as np

FRED_API_KEY = "0e64e6f57e33d91913bedf713d1cf764"

ETFS         = ['SPY', 'EFA', 'IEF', 'VNQ', 'DBC']
MACRO_SERIES = ['FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO', 'TB3MS']

START      = '2006-02-01'
END        = '2024-12-31'

## ALFRED Fetch Function

In [2]:
BASE_URL = 'https://api.stlouisfed.org/fred/series/observations'

def fetch_alfred(series_id, api_key, start, end):
    def _fetch_vintage(rt_start, rt_end):
        url = (
            f"{BASE_URL}?series_id={series_id}"
            f"&realtime_start={rt_start}&realtime_end={rt_end}"
            f"&output_type=2&api_key={api_key}&file_type=json"
        )
        with urllib.request.urlopen(url) as resp:
            return json.loads(resp.read())

    def _parse_wide(data):
        df = pd.DataFrame(data['observations']).set_index('date')
        df.index = pd.to_datetime(df.index)
        df = df.replace('.', np.nan).apply(pd.to_numeric, errors='coerce')
        return df.apply(lambda row: row.dropna().iloc[0] if row.notna().any() else np.nan, axis=1)

    try:
        return _parse_wide(_fetch_vintage(start, end))
    except urllib.error.HTTPError:
        pass

    try:
        years = pd.date_range(start=start, end=end, freq='YS')
        chunks = []
        for i, yr_start in enumerate(years):
            yr_end = (years[i + 1] - pd.Timedelta(days=1)) if i + 1 < len(years) else pd.Timestamp(end)
            chunk_data = _fetch_vintage(yr_start.strftime('%Y-%m-%d'), yr_end.strftime('%Y-%m-%d'))
            chunks.append(_parse_wide(chunk_data))
        return pd.concat(chunks).sort_index()
    except urllib.error.HTTPError:
        pass

    print(f'    {series_id} does not support vintage output. Using FRED instead.')
    url = (
        f"{BASE_URL}?series_id={series_id}"
        f"&observation_start={start}&observation_end={end}"
        f"&api_key={api_key}&file_type=json"
    )
    with urllib.request.urlopen(url) as resp:
        data = json.loads(resp.read())
    df = pd.DataFrame(data['observations'])
    df['date'] = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'].replace('.', np.nan), errors='coerce')
    return df.set_index('date')['value']

## Step 1: Download ETF Prices

In [3]:
#Downloading from Yahoo Finance
raw = yf.download(ETFS, start=START, end=END, auto_adjust=True, progress=False)
prices_daily = raw['Close'][ETFS]

#Searches for missing values and does forward-filling
missing = prices_daily.isnull().sum()
if missing.any():
    prices_daily = prices_daily.ffill()

#Resampling
prices_monthly = prices_daily.resample('ME').last()

prices_daily.to_csv('../cleaned_prices_daily.csv')

## Step 2: Download Macro Indicators from ALFRED

In [4]:
macro_frames = {}
for series_id in MACRO_SERIES:
    point_in_time = fetch_alfred(series_id, FRED_API_KEY, START, END)
    point_in_time = point_in_time.loc[START:END]
    macro_frames[series_id] = point_in_time

macro_daily = pd.DataFrame(macro_frames)
macro_daily.index = pd.to_datetime(macro_daily.index)
macro_daily = macro_daily.resample('D').ffill()
macro_monthly = macro_daily.resample('ME').last()
macro_monthly = macro_monthly.loc[START:END]

    T10Y2Y does not support vintage output. Using FRED instead.


## Step 3: Align Datasets to Common Month-End Index

In [5]:
common_index = prices_monthly.index.intersection(macro_monthly.index)
prices_monthly = prices_monthly.loc[common_index]
macro_monthly  = macro_monthly.loc[common_index]

## Step 4: Outlier Check

In [6]:
returns = prices_monthly.pct_change()
for etf in ETFS:
    col = returns[etf].dropna()
    z_scores = (col - col.mean()) / col.std()
    outliers = col[z_scores.abs() > 4]
    if not outliers.empty:
        print(f'  {etf}: {len(outliers)} extreme return(s) flagged')
        for dt, val in outliers.items():
            print(f'    {dt.date()}: {val:.2%}')
    else:
        print(f'  {etf}: no extreme outliers detected.')

  SPY: no extreme outliers detected.
  EFA: 1 extreme return(s) flagged
    2008-10-31: -20.83%
  IEF: no extreme outliers detected.
  VNQ: 2 extreme return(s) flagged
    2008-10-31: -31.73%
    2009-04-30: 30.68%
  DBC: 1 extreme return(s) flagged
    2008-10-31: -25.08%


## Step 5: Save to Disk

In [7]:
prices_monthly.to_csv('../cleaned_prices.csv')
macro_monthly.to_csv('../cleaned_macro.csv')